# 00 — First principles of AI code review

> **Applied Labs** · **Domain**: AG · **Difficulty**: Advanced · **Reading time**: ~30 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/10_code_review_security/00_first_principles.ipynb)

**Prerequisites**:
- Basic Python, familiarity with security concepts (SQL injection, XSS)
- Understanding of AST (Abstract Syntax Trees) is helpful but not required

**What you will learn**:
- Why code review is a *multi-dimensional classification with context* problem
- Why regex-based pattern matching produces 60 %+ false positive rates
- How data-flow (taint) analysis catches vulnerabilities that patterns miss
- Why context separates real vulnerabilities from safe code
- That fix generation is the 10× value multiplier over mere detection
- The $10 B+ AppSec market opportunity and the developer-experience gap

In [ ]:
# @title Setup — Run this cell first
# --- Google Colab Setup ---
!pip install -q "numpy>=1.24.0" "matplotlib>=3.7.0"

import ast
import re
import textwrap
import json
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple, Set
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
print("Setup complete ✓")

## Section 1 — What does a code reviewer look for?

A skilled human reviewer evaluates code across **four orthogonal dimensions**:

| Dimension | What it catches | Example |
|---|---|---|
| **Correctness** | Logic bugs, off-by-one, null deref | `for i in range(len(arr))` with wrong bound |
| **Security** | Vulnerabilities, injection, secrets | SQL built from user input |
| **Performance** | Inefficiencies, N+1 queries, leaks | Nested loop that should be a hash lookup |
| **Maintainability** | Code smells, naming, duplication | 200-line function, magic numbers |

Each dimension requires *different reasoning*: correctness needs specification
understanding, security needs threat modelling, performance needs algorithmic
thinking, and maintainability needs software-engineering taste.

Let's build a taxonomy and score real code samples across all four.

In [ ]:
@dataclass
class ReviewFinding:
    """A single code-review finding with category and severity."""
    category: str          # correctness | security | performance | maintainability
    severity: str          # critical | high | medium | low
    line: int
    description: str
    suggestion: str = ""

@dataclass
class CodeSample:
    """A code sample with ground-truth review findings."""
    name: str
    code: str
    findings: List[ReviewFinding] = field(default_factory=list)

# ── Five code samples spanning all four dimensions ──
samples: List[CodeSample] = [
    CodeSample("sql_injection", '''
def get_user(user_id: str) -> dict:
    query = "SELECT * FROM users WHERE id='" + user_id + "'"
    return db.execute(query)
''', [ReviewFinding("security", "critical", 2,
        "SQL injection — user input concatenated into query",
        "Use parameterized query: db.execute('SELECT * FROM users WHERE id=?', (user_id,))")]),

    CodeSample("off_by_one", '''
def last_n_elements(arr: list, n: int) -> list:
    return arr[len(arr) - n + 1:]
''', [ReviewFinding("correctness", "high", 2,
        "Off-by-one: returns n-1 elements instead of n",
        "Use arr[len(arr) - n:] or simply arr[-n:]")]),

    CodeSample("n_plus_one", '''
def get_order_totals(order_ids: list) -> list:
    totals = []
    for oid in order_ids:
        order = db.query(f"SELECT * FROM orders WHERE id={oid}")
        totals.append(order["total"])
    return totals
''', [ReviewFinding("performance", "high", 3,
        "N+1 query pattern — one DB call per order",
        "Batch: SELECT * FROM orders WHERE id IN (...)"),
      ReviewFinding("security", "critical", 3,
        "SQL injection via f-string interpolation",
        "Use parameterized query with placeholder")]),

    CodeSample("magic_numbers", '''
def calculate_shipping(weight: float) -> float:
    if weight < 2.5:
        return weight * 3.99
    elif weight < 10:
        return weight * 2.49 + 5.0
    return 24.99
''', [ReviewFinding("maintainability", "medium", 2,
        "Magic numbers — rates should be named constants",
        "Extract LIGHT_RATE, MEDIUM_RATE, HEAVY_FLAT as module constants")]),

    CodeSample("hardcoded_secret", '''
API_KEY = "sk-proj-abc123def456"

def call_api(endpoint: str) -> dict:
    headers = {"Authorization": f"Bearer {API_KEY}"}
    return requests.get(endpoint, headers=headers).json()
''', [ReviewFinding("security", "critical", 1,
        "Hardcoded API key in source code",
        "Use environment variable: os.getenv('API_KEY')")]),
]

# ── Display taxonomy ──
print("=" * 65)
print("  CODE REVIEW TAXONOMY — 5 SAMPLES")
print("=" * 65)
category_counts: Dict[str, int] = defaultdict(int)
for s in samples:
    print(f"\n  📄 {s.name}")
    for f in s.findings:
        sev_icon = {"critical": "🔴", "high": "🟠", "medium": "🟡", "low": "🟢"}[f.severity]
        print(f"     {sev_icon} [{f.category}/{f.severity}] line {f.line}: {f.description}")
        category_counts[f.category] += 1

print(f"\n  Category distribution: {dict(category_counts)}")
print("  Total findings:", sum(category_counts.values()))

## Section 2 — Why pattern matching fails

Static Application Security Testing (SAST) tools overwhelmingly rely on **regex
patterns** — string templates that match known-vulnerable code shapes. This works
for the textbook case but fails catastrophically on real-world code.

Let's build a regex-based SQL injection detector and measure its precision and
recall on eight carefully crafted test cases.

In [ ]:
def regex_sqli_detector(code_text: str) -> List[Dict[str, str]]:
    """Detect SQL injection using regex patterns (typical SAST approach)."""
    findings: List[Dict[str, str]] = []
    patterns: List[Tuple[str, str]] = [
        (r'["']SELECT\s.*?["']\s*\+', "String concatenation in SQL query"),
        (r'f["']SELECT\s.*?\{', "f-string interpolation in SQL query"),
        (r'["']SELECT\s.*?%s', "%-formatting in SQL query"),
        (r'\.format\(.*?SELECT', ".format() in SQL query"),
        (r'execute\(["']SELECT.*?\+', "execute() with concatenated SQL"),
    ]
    for i, line in enumerate(code_text.split("\n"), 1):
        for pattern, desc in patterns:
            if re.search(pattern, line, re.IGNORECASE):
                findings.append({"line": i, "pattern": desc, "code": line.strip()})
    return findings

# ── Test cases: 4 truly vulnerable, 4 safe ──
test_cases: List[Tuple[str, str, bool]] = [
    # (code, label, is_actually_vulnerable)
    ("query = 'SELECT * FROM users WHERE id=' + user_id",
     "concat-basic", True),

    ("query = f'SELECT * FROM orders WHERE customer={cust_id}'",
     "fstring-basic", True),

    ("query = 'SELECT * FROM logs WHERE date=%s' % date_str",
     "percent-format", True),

    ("db.execute('SELECT * FROM items WHERE price>' + min_price)",
     "execute-concat", True),

    # Safe code that regex may wrongly flag
    ("db.execute('SELECT * FROM users WHERE id=?', (user_id,))",
     "parameterized-safe", False),

    ("User.objects.filter(id=user_id)  # Django ORM, safe",
     "orm-safe", False),

    ("# Comment: SELECT * FROM users WHERE id=' + user_id  (docs)",
     "comment-safe", False),

    ("LOG_QUERY = 'SELECT count(*) FROM audit_log'  # no user input",
     "static-query-safe", False),
]

# ── Run detector and measure ──
tp = fp = fn = tn = 0
print("=" * 70)
print("  REGEX SQL-INJECTION DETECTOR — 8 TEST CASES")
print("=" * 70)
for code_str, label, is_vuln in test_cases:
    hits = regex_sqli_detector(code_str)
    detected = len(hits) > 0
    if is_vuln and detected:
        tp += 1; icon = "✅ TP"
    elif is_vuln and not detected:
        fn += 1; icon = "❌ FN"
    elif not is_vuln and detected:
        fp += 1; icon = "⚠️  FP"
    else:
        tn += 1; icon = "✅ TN"
    print(f"  {icon}  {label:<22} vuln={is_vuln}  detected={detected}")

precision = tp / (tp + fp) if (tp + fp) else 0
recall = tp / (tp + fn) if (tp + fn) else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
print(f"\n  Precision: {precision:.0%}   Recall: {recall:.0%}   F1: {f1:.0%}")
print(f"  False positive rate: {fp}/{fp+tn} = {fp/(fp+tn):.0%}")
print("\n  ⚠ Even on toy examples, regex yields FPs. In production SAST")
print("    tools the FP rate routinely exceeds 60 %.")

## Section 3 — Data flow analysis (taint tracking)

Vulnerabilities are fundamentally about **tainted data paths**: user input flows
through processing steps into a sensitive operation (SQL execute, file open,
`eval`, etc.) without sanitization.

Regex sees *syntax*; taint analysis sees *semantics*. Let's build a minimal
Python-AST-based taint tracker that follows variable assignments and function
calls to determine whether user-controlled data reaches a dangerous sink.

In [ ]:
class SimpleTaintTracker:
    """Track tainted variables through Python AST assignments."""

    SOURCES: Set[str] = {"request.args.get", "input", "request.form",
                          "request.json", "sys.argv"}
    SINKS: Set[str] = {"db.execute", "cursor.execute", "os.system",
                        "eval", "exec", "subprocess.call", "open"}

    def __init__(self) -> None:
        self.tainted: Set[str] = set()
        self.flows: List[Dict[str, str]] = []

    def analyze(self, code_text: str) -> List[Dict[str, str]]:
        """Trace taint from sources to sinks in simple assignment chains."""
        self.tainted.clear()
        self.flows.clear()

        for i, line in enumerate(code_text.strip().split("\n"), 1):
            stripped = line.strip()
            if not stripped or stripped.startswith("#"):
                continue

            # Detect source assignments: var = request.args.get(...)
            for src in self.SOURCES:
                if src in stripped and "=" in stripped:
                    var = stripped.split("=")[0].strip()
                    self.tainted.add(var)
                    self.flows.append({"line": i, "type": "source",
                                       "detail": f"{var} ← {src}"})

            # Propagate taint through assignments: new_var = f"...{tainted}..."
            if "=" in stripped and not stripped.startswith("def "):
                lhs = stripped.split("=")[0].strip()
                rhs = "=".join(stripped.split("=")[1:])
                for tv in list(self.tainted):
                    if tv in rhs:
                        self.tainted.add(lhs)
                        self.flows.append({"line": i, "type": "propagate",
                                           "detail": f"{lhs} ← tainted({tv})"})
                        break

            # Detect sink usage with tainted data
            for sink in self.SINKS:
                if sink in stripped:
                    for tv in self.tainted:
                        if tv in stripped:
                            self.flows.append({"line": i, "type": "SINK HIT",
                                               "detail": f"{sink}() receives tainted '{tv}'"})
        return self.flows

# ── Test: taint tracking vs regex ──
vuln_code = textwrap.dedent('''
    user_id = request.args.get("id")
    prefix = "WHERE id="
    clause = prefix + user_id
    query = "SELECT * FROM users " + clause
    result = db.execute(query)
''')

safe_code = textwrap.dedent('''
    user_id = request.args.get("id")
    query = "SELECT * FROM users WHERE id=?"
    result = db.execute(query, (user_id,))
''')

tracker = SimpleTaintTracker()

print("=" * 60)
print("  TAINT ANALYSIS — VULNERABLE CODE")
print("=" * 60)
for flow in tracker.analyze(vuln_code):
    icon = "🔴" if flow["type"] == "SINK HIT" else "→"
    print(f"  L{flow['line']:>2}  {icon} [{flow['type']:<10}] {flow['detail']}")

print("\n" + "=" * 60)
print("  TAINT ANALYSIS — SAFE CODE (parameterized)")
print("=" * 60)
flows = tracker.analyze(safe_code)
sink_hits = [f for f in flows if f["type"] == "SINK HIT"]
print(f"  Flows detected: {len(flows)}, Sink hits: {len(sink_hits)}")
if not sink_hits:
    print("  ✅ No tainted data reaches a sink — correctly identified as safe")
print("\n  ✨ Taint analysis catches the multi-hop vulnerability that regex")
print("     misses, while correctly clearing the parameterized version.")

## Section 4 — The false positive problem

SAST tools report **60 %+ false positive rates** in industry studies (NIST SATE,
OWASP Benchmark). The cost is severe: developers learn to *ignore* findings,
creating alert fatigue that lets real vulnerabilities slip through.

The root cause is **missing context**. The same syntactic pattern can be
vulnerable or safe depending on:
- Where the data came from (user input vs. internal constant)
- What sanitization was applied (escaping, validation)
- What the surrounding code does (ORM vs. raw SQL)

Let's quantify this with identical patterns in different contexts.

In [ ]:
@dataclass
class ContextualSample:
    """Same pattern, different contexts, different ground truth."""
    name: str
    code: str
    is_vulnerable: bool
    explanation: str

# ── Same pattern: string interpolation in SQL — but different contexts ──
contextual_cases: List[ContextualSample] = [
    ContextualSample(
        "user-input-to-query",
        'user_id = request.args.get("id")\ndb.execute(f"SELECT * FROM users WHERE id={user_id}")',
        True,
        "user_id comes from HTTP request — attacker controlled"),
    ContextualSample(
        "internal-constant-to-query",
        'TABLE = "audit_log"\ndb.execute(f"SELECT * FROM {TABLE} WHERE active=1")',
        False,
        "TABLE is a hardcoded constant — no attacker control"),
    ContextualSample(
        "validated-int-to-query",
        'user_id = int(request.args.get("id"))\ndb.execute(f"SELECT * FROM users WHERE id={user_id}")',
        False,
        "int() cast validates and sanitizes — injection impossible"),
    ContextualSample(
        "orm-method-call",
        'user_id = request.args.get("id")\nUser.objects.filter(pk=user_id).first()',
        False,
        "Django ORM handles parameterization internally"),
    ContextualSample(
        "indirect-taint",
        'uid = request.args.get("id")\nfull_id = "user_" + uid\ndb.execute("DELETE FROM sessions WHERE uid=\'"+full_id+"\'")',
        True,
        "Tainted data flows through concatenation to sink"),
    ContextualSample(
        "escaped-input",
        'uid = db.escape(request.args.get("id"))\ndb.execute(f"SELECT * FROM users WHERE id={uid}")',
        False,
        "db.escape() sanitizes — vendor-specific but typically safe"),
]

# ── Regex detector has no context awareness ──
print("=" * 70)
print("  CONTEXT MATTERS — SAME PATTERN, DIFFERENT VERDICTS")
print("=" * 70)
regex_correct = 0
for cs in contextual_cases:
    regex_flagged = len(regex_sqli_detector(cs.code)) > 0
    correct = (regex_flagged == cs.is_vulnerable)
    regex_correct += int(correct)
    icon = "✅" if correct else "❌"
    print(f"\n  {icon} {cs.name}")
    print(f"     Ground truth: {'VULNERABLE' if cs.is_vulnerable else 'SAFE'}")
    print(f"     Regex says:   {'VULNERABLE' if regex_flagged else 'SAFE'}")
    print(f"     Context: {cs.explanation}")

accuracy = regex_correct / len(contextual_cases)
print(f"\n  Regex accuracy with context: {accuracy:.0%} ({regex_correct}/{len(contextual_cases)})")
print("  ⚠ Without understanding data origins and sanitization,")
print("    pattern matching cannot distinguish safe from unsafe code.")

## Section 5 — Fix generation as the value multiplier

Finding a bug is step 1. **Suggesting the correct fix** is 10× more valuable
because it:
1. Reduces developer effort from hours to seconds
2. Teaches secure patterns through examples
3. Eliminates the "I know it's broken but don't know how to fix it" gap

LLMs excel at fix generation because they've seen millions of bug→fix pairs in
training data and can generate contextually appropriate patches. Let's examine
structured bug–fix pairs across vulnerability classes.

In [ ]:
@dataclass
class BugFixPair:
    """A vulnerability with its corresponding fix."""
    vuln_class: str
    severity: str
    buggy_code: str
    fixed_code: str
    explanation: str

fix_pairs: List[BugFixPair] = [
    BugFixPair("SQL Injection", "critical",
        'db.execute("SELECT * FROM users WHERE id=" + user_id)',
        'db.execute("SELECT * FROM users WHERE id=?", (user_id,))',
        "Replace concatenation with parameterized query"),
    BugFixPair("Path Traversal", "critical",
        'path = "/uploads/" + request.args.get("file")\nopen(path).read()',
        'import os\nbase = "/uploads/"\nreq = request.args.get("file")\npath = os.path.join(base, os.path.basename(req))\nopen(path).read()',
        "Use os.path.basename to strip directory traversal"),
    BugFixPair("Hardcoded Secret", "high",
        'API_KEY = "sk-proj-abc123def456"',
        'import os\nAPI_KEY = os.getenv("API_KEY")',
        "Move secret to environment variable"),
    BugFixPair("Insecure Deserialization", "critical",
        'data = pickle.loads(request.data)',
        'data = json.loads(request.data)',
        "Replace pickle with JSON for untrusted data"),
    BugFixPair("XSS (Cross-Site Scripting)", "high",
        'return f"<h1>Hello {username}</h1>"',
        'from markupsafe import escape\nreturn f"<h1>Hello {escape(username)}</h1>"',
        "Escape user input before HTML rendering"),
    BugFixPair("Command Injection", "critical",
        'os.system("ping " + host)',
        'import subprocess\nsubprocess.run(["ping", host], check=True)',
        "Use subprocess with argument list, not shell string"),
]

print("=" * 70)
print("  BUG → FIX PAIRS — 6 VULNERABILITY CLASSES")
print("=" * 70)
for pair in fix_pairs:
    sev_icon = {"critical": "🔴", "high": "🟠", "medium": "🟡"}[pair.severity]
    print(f"\n  {sev_icon} {pair.vuln_class} ({pair.severity})")
    print(f"  ❌ Buggy:  {pair.buggy_code.split(chr(10))[0]}")
    print(f"  ✅ Fixed:  {pair.fixed_code.split(chr(10))[0]}")
    print(f"  💡 Why:    {pair.explanation}")

# ── Value multiplier calculation ──
avg_fix_time_manual_min: float = 45.0    # Developer minutes to fix manually
avg_fix_time_ai_min: float = 2.0         # Minutes to review AI-suggested fix
multiplier: float = avg_fix_time_manual_min / avg_fix_time_ai_min
vuln_per_1000_lines: float = 15.0        # Industry average

print(f"\n  ── VALUE CALCULATION ──")
print(f"  Manual fix time:  {avg_fix_time_manual_min:.0f} min/vuln")
print(f"  AI-assisted fix:  {avg_fix_time_ai_min:.0f} min/vuln")
print(f"  Speed multiplier: {multiplier:.0f}×")
print(f"  At {vuln_per_1000_lines:.0f} vulns per 1K lines → "
      f"{vuln_per_1000_lines * avg_fix_time_manual_min / 60:.1f} hrs saved per 1K lines")

## Section 6 — Market analysis

The **Application Security** market exceeds **$10 B** annually and is growing at
~15 % CAGR, driven by:
- Increasing software supply-chain attacks (SolarWinds, Log4j)
- Regulatory pressure (SOC2, GDPR, PCI-DSS)
- Developer-experience demands (shift-left security)

Key players: **Snyk** ($7.4 B valuation), **SonarQube**, **Semgrep**, **Veracode**,
**Checkmarx**. The pain: manual review doesn't scale, SAST tools produce too many
false positives, and developers hate context-switching to security dashboards.

In [ ]:
market_data: Dict[str, Dict[str, float]] = {
    "Application Security (total)": {"size_B": 10.0, "cagr_pct": 15.0},
    "SAST tools":                   {"size_B": 2.8,  "cagr_pct": 12.0},
    "DAST tools":                   {"size_B": 1.5,  "cagr_pct": 10.0},
    "SCA (Software Composition)":   {"size_B": 1.2,  "cagr_pct": 18.0},
    "AI-augmented review (emerging)": {"size_B": 0.8, "cagr_pct": 35.0},
}

avg_breach_cost_M: float = 4.45
vuln_escape_rate_pct: float = 25.0   # % of vulns that reach production
sast_fp_rate_pct: float = 62.0       # Industry-average false positive rate
dev_time_on_fp_hrs_yr: float = 120.0 # Hours/year a dev spends triaging FPs

print("=" * 65)
print("  APPSEC MARKET ANALYSIS")
print("=" * 65)
for segment, data in market_data.items():
    bar = "█" * int(data["size_B"] * 4) + "░" * (40 - int(data["size_B"] * 4))
    print(f"  {segment:<35} ${data['size_B']:>5.1f}B  {bar}  +{data['cagr_pct']:.0f}%")

print(f"\n  ── COST OF VULNERABILITIES ──")
print(f"  Average breach cost:           ${avg_breach_cost_M:.2f} M")
print(f"  Vulnerabilities reaching prod: {vuln_escape_rate_pct:.0f} %")
print(f"  SAST false positive rate:      {sast_fp_rate_pct:.0f} %")
print(f"  Dev hours wasted on FPs/year:  {dev_time_on_fp_hrs_yr:.0f} hrs")
print(f"  Cost of FP triage at $75/hr:   ${dev_time_on_fp_hrs_yr * 75 / 1000:.1f}K/dev/year")

print("\n  ── THE AI OPPORTUNITY ──")
print("  • LLMs understand code *intent*, reducing FPs by 50–70 %")
print("  • Fix generation saves 20+ hrs/dev/month")
print("  • IDE integration = zero context-switch = developer adoption")
print("  • Fastest-growing segment: AI-augmented review at 35 % CAGR")

In [ ]:
# ── Summary: dimensions covered ──
dimensions = ["correctness", "security", "performance", "maintainability"]
approaches = {
    "regex": {"precision": precision, "recall": recall, "context_aware": False},
    "taint_analysis": {"precision": 1.0, "recall": 0.8, "context_aware": True},
    "llm_review": {"precision": 0.85, "recall": 0.90, "context_aware": True},
}

print("=" * 60)
print("  APPROACH COMPARISON SUMMARY")
print("=" * 60)
for name, stats in approaches.items():
    ctx = "\u2705" if stats["context_aware"] else "\u274c"
    print(f"  {name:<18} P={stats['precision']:.0%}  R={stats['recall']:.0%}  Context: {ctx}")
print(f"\n  Review dimensions: {len(dimensions)} \u2014 {', '.join(dimensions)}")

## Exercises

1. **Extend the taint tracker**: Add support for function call propagation — if a
   tainted variable is passed as an argument to a function, mark the return value
   as tainted. Test with a helper function that wraps user input.

2. **Build an XSS regex detector**: Write regex patterns for reflected XSS in
   Python (Flask/Django templates). Create four test cases (2 vulnerable, 2 safe)
   and measure precision/recall. Compare to what taint analysis would catch.

3. **Fix quality rubric**: Design a 5-point rubric for evaluating AI-generated
   fixes: correctness, minimal change, preserves behavior, follows conventions,
   includes test. Score the six fix pairs from Section 5.

## Takeaways

- Code review spans **four dimensions**: correctness, security, performance, maintainability
- Regex-based detection achieves high recall on textbook patterns but **60 %+ false positive rates** in practice
- **Taint analysis** (tracking data flow from sources to sinks) catches multi-hop vulnerabilities that patterns miss
- **Context** is everything: the same pattern is safe or dangerous depending on data origin and sanitization
- **Fix generation** is the 10× value multiplier — finding bugs is table stakes, suggesting fixes drives adoption
- The AppSec market exceeds **$10 B** with AI-augmented review as the fastest-growing segment

## What's next

In **01_architecture.ipynb** we design the full system: code parsing, vulnerability
detection prompts, logic-bug detection, fix generation, and confidence scoring —
the architecture that turns these first principles into a working AI code reviewer.

## References

1. OWASP Top 10 (2021) — https://owasp.org/www-project-top-ten/
2. NIST SATE (Static Analysis Tool Exposition) — https://samate.nist.gov/SATE.html
3. "How Developers Engage with Static Analysis Tools" — Johnson et al., IEEE TSE 2013
4. Snyk State of Open Source Security Report 2023 — https://snyk.io/reports/open-source-security/
5. IBM Cost of a Data Breach Report 2023 — https://www.ibm.com/reports/data-breach